### Data Splitting
The data proportion would be 70% training, 15% validation, 15% testing

In [1]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split


#### Configuration
---

In [2]:

ORIGINAL_DATASET = r"C:\Users\USER\Documents\Thesis Dataset\Original Dataset"

# Defining path variables for splitting the data
BASE_OUTPUT = r"C:\Users\USER\Documents\Thesis Dataset\Processed Dataset"
TRAIN_DIR = os.path.join(BASE_OUTPUT, "train")
VAL_DIR   = os.path.join(BASE_OUTPUT, "val")
TEST_DIR  = os.path.join(BASE_OUTPUT, "test")

# Splitting percentage or proportion
train_ratio = 0.70
val_ratio   = 0.15
test_ratio  = 0.15

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")

#### Clean and Create Directories 

Remove old processed data to avoid duplicates and create paths for training, validation and testing

In [3]:

# Deletes an already existed file
if os.path.exists(BASE_OUTPUT):
    shutil.rmtree(BASE_OUTPUT)

# Creates the directory for the path variables
for dir_path in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(dir_path, exist_ok=True)


#### SPLIT & COPY FUNCTION

In [4]:

def split_and_copy(class_images, train_path, val_path, test_path):
    """Split the list of image paths into train/val/test and copy once."""
    if len(class_images) < 3:
        print(f"Folder is Empty or does not have an image: {os.path.basename(train_path)}")
        return

    random.shuffle(class_images)

    # Splitting Data: Train vs Temp (Val+Test)
    train_imgs, temp_imgs = train_test_split(
        class_images, test_size=(1 - train_ratio), random_state=42
    )
    # Splitting the Temp: Validation and Test
    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=(test_ratio / (test_ratio + val_ratio)),
        random_state=42,
    )

    # Copy each image
    for img in train_imgs:
        shutil.copy2(img, train_path)
    for img in val_imgs:
        shutil.copy2(img, val_path)
    for img in test_imgs:
        shutil.copy2(img, test_path)


#### Main loop
Loop for each plant to be splitted and copy to the processed folder

split and copy function is used

In [5]:
for disease_class in sorted(os.listdir(ORIGINAL_DATASET)):
    class_path = os.path.join(ORIGINAL_DATASET, disease_class)
    if not os.path.isdir(class_path):
        continue

    images = [
        os.path.join(class_path, f)
        for f in os.listdir(class_path)
        if f.lower().endswith(IMG_EXTENSIONS)
    ]

    train_out = os.path.join(TRAIN_DIR, disease_class)
    val_out = os.path.join(VAL_DIR, disease_class)
    test_out = os.path.join(TEST_DIR, disease_class)

    os.makedirs(train_out, exist_ok=True)
    os.makedirs(val_out, exist_ok=True)
    os.makedirs(test_out, exist_ok=True)

    split_and_copy(images, train_out, val_out, test_out)

    print(f"{disease_class}: {len(images)} images processed ✅")

print("🎉 Dataset successfully split into 70% Train, 15% Validation, 15% Test!")

Banana_Cordana: 442 images processed ✅
Banana_Healthy: 1108 images processed ✅
Banana_Panama Disease: 835 images processed ✅
Banana_Pestalotiopsis: 117 images processed ✅
Banana_Yellow and Black Sigatoka: 2597 images processed ✅
Coconut_Bud Root Dropping: 514 images processed ✅
Coconut_Bud Rot: 470 images processed ✅
Coconut_Gray Leaf Spot: 2135 images processed ✅
Coconut_Healthy: 123 images processed ✅
Coconut_Leaf Rot: 1673 images processed ✅
Coconut_WCLWD_DryingofLeaflets: 1078 images processed ✅
Coconut_WCLWD_Flaccidity: 1069 images processed ✅
Coconut_WCLWD_Yellowing: 1084 images processed ✅
Sugarcane_Healthy: 522 images processed ✅
Sugarcane_Mosaic: 462 images processed ✅
Sugarcane_RedRot: 518 images processed ✅
Sugarcane_Rust: 514 images processed ✅
Sugarcane_Yellow: 505 images processed ✅
🎉 Dataset successfully split into 70% Train, 15% Validation, 15% Test!


### Data Augmentation
We will manipulate the data using **horizontal flip**, **vertical flip**, **brighten**, **darken**, **rotate**, and **contrast**.

In [6]:
import os
import random
from pathlib import Path
from PIL import Image, ImageEnhance, ImageOps, ImageDraw
import numpy as np

#### ⚙️ Configurationn

In [7]:
# Original training set folder directory
TRAIN_DIR = r"C:\Users\USER\Documents\Thesis Dataset\Processed Dataset\train"

# New folder for augmented images
AUGMENT_DIR = r"C:\Users\USER\Documents\Thesis Dataset\Processed Dataset\train_augmented"

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")

# --- AUGMENTATION CONTROLS ---
AUGMENTATION_FACTOR = 7 
MIN_AUG_PER_COPY = 3
MAX_AUG_PER_COPY = 5


# Augmentation parameter ranges
ROTATE_ANGLE = 25
BRIGHTNESS_RANGE = (0.7, 1.4)
CONTRAST_RANGE = (0.7, 1.4)
GAMMA_RANGE = (0.8, 1.4)
NOISE_STD = 10
PERSPECTIVE_DISTORTION_SCALE = 0.2

### 🧩 Data Augmentation Functions
Below are the different augmentation techniques used to diversify the training dataset:

| Function | Description | Simulated Effect |
|---------|-------------|------------------|
| **horizontal_flip ↔️** | Flips the image horizontally (left ↔ right). | Different angle/side of the plant is photographed. |
| **random_brightness ☀️** | Randomly adjusts brightness within a controlled range. | Variation in ambient light or exposure. |
| **random_contrast 🎨** | Randomly adjusts the contrast. | Varying image clarity due to distance or camera settings. |
| **gamma_correction 🔆** | Applies gamma adjustment (non-linear color shift). | Variation in lighting conditions (e.g., shadows, glare). |
| **add_noise 🔊** | Adds Gaussian noise to the image pixels. | Imperfections and variations in camera sensor data. |
| **rotate 🔄** | Rotates the image by up to ±25 degrees. | Slight camera tilt or plant orientation changes. |
| **random_perspective 📐** | Applies a random 4-point perspective distortion. | Simulates images taken from different angles (not directly overhead). |


> 💡 **Tip:**  
> Applying these augmentations helps the model generalize better and handle variations in lighting, orientation, noise, and texture.


In [11]:
def horizontal_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT)

def random_brightness(img):
    factor = random.uniform(*BRIGHTNESS_RANGE)
    return ImageEnhance.Brightness(img).enhance(factor)

def random_contrast(img):
    factor = random.uniform(*CONTRAST_RANGE)
    return ImageEnhance.Contrast(img).enhance(factor)

def rotate(img):
    angle = random.uniform(-ROTATE_ANGLE, ROTATE_ANGLE)
    return img.rotate(angle, expand=True).resize(img.size)

def gamma_correction(img):
    gamma = random.uniform(*GAMMA_RANGE)
    inv_gamma = 1.0 / gamma
    lut = [pow(i / 255, inv_gamma) * 255 for i in range(256)]
    lut = lut * 3
    return img.point(lut)

def add_noise(img):
    arr = np.array(img)
    noise = np.random.normal(0, NOISE_STD, arr.shape).astype(np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def random_perspective(img):
    width, height = img.size
    
    # Define corners (top_left, top_right, bottom_right, bottom_left)
    original_points = [(0, 0), (width, 0), (width, height), (0, height)]
    
    scale = PERSPECTIVE_DISTORTION_SCALE
    
    def get_offset(coord):
        return int(random.uniform(-scale * coord, scale * coord))

    new_points = []
    for x, y in original_points:
        dx = get_offset(width)
        dy = get_offset(height)
        
        new_x = np.clip(x + dx, 0, width)
        new_y = np.clip(y + dy, 0, height)
        new_points.append((new_x, new_y))

    # Flatten points list for the transform function: [x1, y1, x2, y2, ..., x4, y4]
    # Source points (orig) and Destination points (new) must be concatenated
    transform_data = [coord for point in original_points for coord in point]
    transform_data += [coord for point in new_points for coord in point]
    
    # Use the appropriate resampling filter (Handle version difference)
    try:
        resample_filter = Image.Resampling.BICUBIC
    except AttributeError:
        resample_filter = Image.BICUBIC 

    # FIX: Use the standard Image.PERSPECTIVE constant.
    # The transform function calculates the matrix internally when given the 16 points.
    return img.transform(
        img.size, 
        Image.PERSPECTIVE, # Use Image.PERSPECTIVE constant
        transform_data,    # Pass all 16 points (orig and new)
        resample=resample_filter
    )


# ==========================================================
# LIST OF AUGMENTATIONS TO BE RANDOMLY SELECTED (Now 7 functions)
# ==========================================================
RANDOM_AUGMENTATIONS = [
    horizontal_flip,
    random_brightness,
    random_contrast,
    gamma_correction,
    add_noise,
    rotate,
    random_perspective,
]

#### Folders for the augmented data

In [12]:

for root, dirs, _ in os.walk(TRAIN_DIR):
    rel_path = os.path.relpath(root, TRAIN_DIR)
    os.makedirs(os.path.join(AUGMENT_DIR, rel_path), exist_ok=True)

count_original = 0
count_augmented = 0

#### The augmentation process loop

In [ ]:
print(f"🔄 Starting augmentation: Generating {AUGMENTATION_FACTOR} random copies per image...")

for root, _, files in os.walk(TRAIN_DIR):
    for file in files:
        if file.lower().endswith(IMG_EXTENSIONS):

            src = os.path.join(root, file)
            img = Image.open(src).convert("RGB")
            count_original += 1

            rel_path = os.path.relpath(root, TRAIN_DIR)
            save_base = os.path.join(AUGMENT_DIR, rel_path)
            base_name, ext = os.path.splitext(file)
            
            img.save(os.path.join(save_base, file))

            for i in range(AUGMENTATION_FACTOR):
                # Start with a clean copy of the original image
                aug_img = img.copy()
                
                # Determine how many random augmentations to apply
                num_augs = random.randint(MIN_AUG_PER_COPY, MAX_AUG_PER_COPY)
                
                # Randomly select and apply a subset of functions
                applied_funcs = random.sample(RANDOM_AUGMENTATIONS, num_augs)

                for aug_func in applied_funcs:
                    aug_img = aug_func(aug_img)

                # Save the randomly augmented image with a unique name
                aug_img.save(os.path.join(save_base, f"{base_name}_aug{i}{ext}"))
                count_augmented += 1

total_images = count_original + count_augmented
expected_images = count_original * (AUGMENTATION_FACTOR + 1) 

print("\n🍀 Data Augmentation Complete!")
print(f"Original Images: {count_original}")
print(f"Augmented Images Generated: {count_augmented}")
print(f"Total Images in '{os.path.basename(AUGMENT_DIR)}': {total_images} (Expected: {expected_images})")
print(f"📁 Saved to: {AUGMENT_DIR}")

🔄 Starting augmentation: Generating 7 random copies per image...


#### TESTING AND PRINTING IMAGES

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

TRAIN_AUGMENTED_DIR = AUGMENT_DIR
IMG_SIZE = (224, 224)

def show_samples_per_class(directory):
    for disease_class in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, disease_class)
        if not os.path.isdir(class_path):
            continue

        print(f"\n🌿 Class: {disease_class}")
        img_files = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(IMG_EXTENSIONS)
        ]
        if not img_files:
            print("No images found. Skipping.")
            continue

        plt.figure(figsize=(4, 4))
        img_file = random.choice(img_files)
        img_path = os.path.join(class_path, img_file)

        img = tf.io.read_file(img_path)
        img = tf.image.decode_image(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.uint8)

        plt.imshow(img.numpy())
        plt.title(disease_class)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

show_samples_per_class(TRAIN_DIR)